# ResumeAI — Production v3: Hardened Fine-Tuning Run

## What this notebook improves over Production v2

| Improvement | v2 | v3 (this notebook) |
|---|---|---|
| **Statistical rigor** | Single training run — Platt vs isotonic ordering flips between reruns | 3 seeds, metrics reported as mean ± std; bootstrap CI proves the calibrator tie |
| **Baseline** | No base-model number on the final test | Base MPNet evaluated on the same 106 held-out pairs → honest before/after delta |
| **GPU budget** | Retrains RoBERTa + DistilRoBERTa cross-encoders that notebooks 03–04 already proved lose | Trains only what ships: the production bi-encoder |
| **Recruiter metric** | Spearman/MAE only | Adds precision@1: "does the top-ranked resume match the recruiter's true best pick?" |
| **Deployment** | Manual push instructions | Publishes model + both calibrators to HF Hub with an auto-generated model card |
| **Reproducibility** | Scattered constants | Single CONFIG dict; per-seed determinism; results exported as JSON |

**Calibrator decision (locked in from the v2 rerun analysis):** Platt scaling is the
production calibrator. Isotonic scores identically within noise, but its non-parametric
step function can overfit a 106-pair calibration split; Platt's 2-parameter sigmoid cannot.
This notebook quantifies that tie with a bootstrap instead of hand-waving it.

**Runtime:** T4 GPU, ~60–75 min (3 training runs @ ~20 min). Upload the two CSVs when prompted.


In [ ]:
# ============================================================
# CELL 1: Config, dependencies, GPU check
# ============================================================
!pip install -q sentence-transformers==3.* datasets pandas scikit-learn matplotlib huggingface_hub

CONFIG = {
    "seeds": [42, 43, 44],          # multi-seed training for mean ± std reporting
    "epochs": 4,
    "batch_size": 16,
    "n_augment": 3,                 # augmented copies per training pair
    "max_jd_words": 350,
    "split_seed": 42,               # fixed: keeps cal/test split identical across seeds & reruns
    "val_fraction": 0.15,
    "bootstrap_resamples": 2000,
    "hf_repo": "dlepighe1/resume-jd-matcher-mpnet",   # <- your HF username/repo
}

import torch, time, json, pickle, warnings, re, random
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from collections import Counter
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.isotonic import IsotonicRegression
from scipy.stats import spearmanr
from scipy.optimize import minimize

from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB)")
else:
    print("WARNING: No GPU! Runtime -> Change runtime type -> T4 GPU")
print(f"\nCONFIG: {json.dumps(CONFIG, indent=2)}")


## 1. Data

Upload `resume_jd_training_800.csv` (815 pairs / 305 JDs) and
`external_test_200_pairs.csv` (212 pairs / 53 completely unseen JDs).

The external set splits 106 **calibration** / 106 **final test** with a fixed seed,
so the final-test pairs are identical across all training seeds and across reruns —
per-seed differences in the results are attributable to training, not to the split.


In [ ]:
# ============================================================
# CELL 2: Upload and split data
# ============================================================
from google.colab import files

print("Upload TRAINING data (resume_jd_training_800.csv):")
up1 = files.upload()
df = pd.read_csv(list(up1.keys())[0])
print(f"Training: {len(df)} pairs")

print("\nUpload EXTERNAL TEST data (external_test_200_pairs.csv):")
up2 = files.upload()
ext_full = pd.read_csv(list(up2.keys())[0])
print(f"External: {len(ext_full)} pairs")

ext_cal, ext_test = train_test_split(
    ext_full, test_size=0.5, random_state=CONFIG["split_seed"], stratify=ext_full["match_type"]
)
print(f"\nExternal calibration: {len(ext_cal)} | External final test: {len(ext_test)}")
print(f"Training distribution: {dict(Counter(df['match_type']))}")
print(f"Final test distribution: {dict(Counter(ext_test['match_type']))}")


## 2. Preprocessing, augmentation, evaluation framework

Identical preprocessing to v2 (smart JD truncation prioritizing requirements sections;
3× augmentation via section shuffling, sentence dropping, keyword noise). Defined once,
before any model is touched, so every seed and every baseline is evaluated identically.


In [ ]:
# ============================================================
# CELL 3: Preprocessing + augmentation (same logic as production v2)
# ============================================================

def smart_truncate_jd(jd_text, max_words=350):
    boilerplate = [r'equal opportunity employer.*', r'does not discriminate.*',
        r'reasonable accommodation.*', r'(compensation|salary|pay) range.*',
        r'(what we offer|our benefits|perks and benefits|benefits:).*']
    cleaned = jd_text
    for p in boilerplate:
        cleaned = re.split(p, cleaned, flags=re.IGNORECASE)[0]
    req = [r'(requirements?|qualifications?|what you.?ll need|must have)',
           r'(responsibilities|what you.?ll do|in this role|you will)']
    best = 0
    for p in req:
        m = re.search(p, cleaned, re.IGNORECASE)
        if m:
            s = max(0, m.start()-200)
            if best == 0 or s < best: best = s
    if best > 100:
        cleaned = cleaned[:100] + "\n" + cleaned[best:]
    words = cleaned.split()
    return ' '.join(words[:max_words]).strip() if len(words) > max_words else cleaned.strip()

for d in [df, ext_cal, ext_test, ext_full]:
    d['jd_clean'] = d['jd'].apply(lambda x: smart_truncate_jd(x, CONFIG["max_jd_words"]))

def shuffle_sections(text):
    sections = {}; curr = 'header'; lines = []
    for line in text.split('\n'):
        u = line.strip().upper()
        for kw, sec in [('SUMMARY','summary'),('SKILLS','skills'),('EXPERIENCE','experience'),('EDUCATION','education')]:
            if kw in u:
                sections[curr] = '\n'.join(lines); curr = sec; lines = [line]; break
        else:
            lines.append(line)
    sections[curr] = '\n'.join(lines)
    header = sections.get('header','')
    body = ['summary','skills','experience','education']
    random.shuffle(body)
    return (header + ''.join(sections.get(s,'') for s in body if s in sections)).strip()

def drop_sentences(text, rate=0.12):
    sents = re.split(r'(?<=[.!?])\s+|\n', text)
    if len(sents) <= 3: return text
    drop = set(random.sample(range(1,len(sents)), min(max(1,int(len(sents)*rate)), len(sents)-2)))
    return ' '.join(s for i,s in enumerate(sents) if i not in drop).strip()

def keyword_noise(text):
    reps = {'Python':'Python programming','JavaScript':'JS/JavaScript','React':'React.js',
            'AWS':'Amazon Web Services','SQL':'SQL databases','Docker':'Docker containers',
            '5+ years':'five or more years','3+ years':'three or more years'}
    r = text
    for k in random.sample(list(reps.keys()), min(2,len(reps))):
        if k in r: r = r.replace(k, reps[k], 1)
    return r

def augment_dataset(dataframe, n_aug, rng):
    rows = []
    aug_types = ['shuffle','drop_jd','drop_resume','noise']
    for _, row in dataframe.iterrows():
        rows.append({'resume':row['resume'],'jd':row['jd_clean'],'score':row['score'],
                     'match_type':row['match_type'],'augmented':False})
        for aug in random.sample(aug_types, min(n_aug, len(aug_types))):
            r, j, s = row['resume'], row['jd_clean'], row['score']
            if aug == 'shuffle': r = shuffle_sections(r)
            elif aug == 'drop_jd': j = drop_sentences(j)
            elif aug == 'drop_resume': r = drop_sentences(r, 0.08)
            elif aug == 'noise': r = keyword_noise(r); j = keyword_noise(j)
            s_noisy = float(np.clip(s + rng.uniform(-0.02, 0.02), 0.01, 0.99))
            rows.append({'resume':r,'jd':j,'score':s_noisy,'match_type':row['match_type'],'augmented':True})
    return pd.DataFrame(rows)

print("Preprocessing + augmentation ready")


In [ ]:
# ============================================================
# CELL 4: Evaluation framework + Platt calibrator
# ============================================================

class PlattCalibrator:
    """2-parameter sigmoid: calibrated = 1/(1+exp(-(a*raw+b))). Can't overfit small samples."""
    def __init__(self):
        self.a, self.b = 1.0, 0.0
    def fit(self, raw_scores, true_scores):
        raw, true = np.asarray(raw_scores), np.asarray(true_scores)
        def loss(params):
            a, b = params
            return float(np.mean((1.0/(1.0+np.exp(-(a*raw+b))) - true)**2))
        res = minimize(loss, x0=[1.0, 0.0], method='Nelder-Mead')
        self.a, self.b = res.x
        return self
    def __call__(self, scores):
        s = np.asarray(scores)
        return (1.0/(1.0+np.exp(-(self.a*s+self.b)))).tolist()

def raw_scores(model, eval_df):
    r = model.encode(eval_df['resume'].tolist(), show_progress_bar=False, convert_to_numpy=True)
    j = model.encode(eval_df['jd_clean'].tolist(), show_progress_bar=False, convert_to_numpy=True)
    # row-wise cosine
    r_n = r / np.linalg.norm(r, axis=1, keepdims=True)
    j_n = j / np.linalg.norm(j, axis=1, keepdims=True)
    return (r_n * j_n).sum(axis=1).tolist()

def metrics(true, pred):
    sp, _ = spearmanr(true, pred)
    mae = float(np.mean(np.abs(np.asarray(true) - np.asarray(pred))))
    return {"spearman": round(float(sp), 4), "mae": round(mae, 4)}

def error_by_type(eval_df, pred, label=""):
    a = eval_df.copy(); a['pred'] = pred; a['err'] = abs(a['score'].astype(float) - a['pred'])
    print(f"\n  {label}")
    print(f"  {'Type':<16} {'MAE':>7} {'True':>7} {'Pred':>7} {'n':>4}")
    for mt in ['strong','good','partial','hard_negative','weak']:
        s = a[a['match_type']==mt]
        if len(s): print(f"  {mt:<16} {s['err'].mean():>7.4f} {s['score'].astype(float).mean():>7.3f} {s['pred'].mean():>7.3f} {len(s):>4}")

test_true = ext_test['score'].astype(float).tolist()
cal_true  = ext_cal['score'].astype(float).tolist()
print("Evaluation framework ready")


## 3. Base-model baseline

The number every fine-tuning claim should be measured against: raw `all-mpnet-base-v2`
on the exact same 106 held-out pairs. v2 never reported this, so the fine-tuning delta
on the headline metric was implicit. Now it's explicit.


In [ ]:
# ============================================================
# CELL 5: Base MPNet on the final test (before fine-tuning)
# ============================================================
base_model = SentenceTransformer('all-mpnet-base-v2')
base_raw = raw_scores(base_model, ext_test)
baseline = metrics(test_true, base_raw)
print(f"BASELINE  all-mpnet-base-v2 (raw cosine): Spearman={baseline['spearman']:.4f} | MAE={baseline['mae']:.4f}")
del base_model; torch.cuda.empty_cache()


## 4. Multi-seed fine-tuning

Three independent training runs (seeds 42/43/44). Each run: re-augment with the seed's
RNG → fresh MPNet → combined CoSENT + CosineSimilarity loss → fit Platt and isotonic on
the calibration split → evaluate on the untouched final test.

Reporting mean ± std across seeds is what turns "my model got 0.86" into "this training
*recipe* reliably produces 0.86 ± x" — the difference between a lucky run and a result.


In [ ]:
# ============================================================
# CELL 6: Train one model per seed
# ============================================================
train_df, val_df = train_test_split(
    df, test_size=CONFIG["val_fraction"], random_state=CONFIG["split_seed"], stratify=df["match_type"]
)
train_df, val_df = train_df.copy(), val_df.copy()
print(f"Train {len(train_df)} | Val {len(val_df)}\n")

seed_runs = []   # one entry per seed: raw/platt/iso metrics + calibrators + model path

for seed in CONFIG["seeds"]:
    print(f"{'='*70}\nSEED {seed}\n{'='*70}")
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    rng = np.random.default_rng(seed)

    aug_df = augment_dataset(train_df, CONFIG["n_augment"], rng)
    print(f"  Augmented: {len(train_df)} -> {len(aug_df)} examples")

    model = SentenceTransformer('all-mpnet-base-v2')
    examples = [InputExample(texts=[r['resume'], r['jd']], label=float(r['score']))
                for _, r in aug_df.iterrows()]
    dl_cosent = DataLoader(examples, shuffle=True, batch_size=CONFIG["batch_size"])
    dl_cosine = DataLoader(examples, shuffle=True, batch_size=CONFIG["batch_size"])
    evaluator = EmbeddingSimilarityEvaluator(
        sentences1=val_df['resume'].tolist(), sentences2=val_df['jd_clean'].tolist(),
        scores=val_df['score'].astype(float).tolist(), name='val')

    start = time.time()
    model.fit(
        train_objectives=[(dl_cosent, losses.CoSENTLoss(model=model)),
                          (dl_cosine, losses.CosineSimilarityLoss(model=model))],
        evaluator=evaluator, epochs=CONFIG["epochs"],
        warmup_steps=int(len(dl_cosent) * 0.1),
        evaluation_steps=len(dl_cosent),
        output_path=f"models/seed{seed}",
        show_progress_bar=True, use_amp=True)
    train_min = (time.time() - start) / 60

    # Calibrate on ext_cal, evaluate on ext_test
    cal_raw = raw_scores(model, ext_cal)
    platt = PlattCalibrator().fit(cal_raw, cal_true)
    iso = IsotonicRegression(out_of_bounds='clip').fit(cal_raw, cal_true)

    t_raw = raw_scores(model, ext_test)
    run = {
        "seed": seed,
        "train_minutes": round(train_min, 1),
        "raw": metrics(test_true, t_raw),
        "platt": metrics(test_true, platt(t_raw)),
        "isotonic": metrics(test_true, iso.predict(t_raw).tolist()),
        "_platt_obj": platt, "_iso_obj": iso, "_test_raw": t_raw,
        "model_path": f"models/seed{seed}",
    }
    seed_runs.append(run)
    print(f"  [{train_min:.0f} min]  raw {run['raw']}  |  platt {run['platt']}  |  iso {run['isotonic']}")
    del model; torch.cuda.empty_cache()

print("\nAll seeds complete")


In [ ]:
# ============================================================
# CELL 7: Aggregate across seeds — mean ± std, pick production seed
# ============================================================
def agg(key, metric):
    vals = [r[key][metric] for r in seed_runs]
    return float(np.mean(vals)), float(np.std(vals))

print(f"{'Variant':<12} {'Spearman':>18} {'MAE':>18}")
print('-'*50)
rows_agg = {}
for key in ['raw', 'platt', 'isotonic']:
    sp_m, sp_s = agg(key, 'spearman'); mae_m, mae_s = agg(key, 'mae')
    rows_agg[key] = {"spearman_mean": round(sp_m,4), "spearman_std": round(sp_s,4),
                     "mae_mean": round(mae_m,4), "mae_std": round(mae_s,4)}
    print(f"{key:<12} {sp_m:>10.4f} ± {sp_s:.4f} {mae_m:>10.4f} ± {mae_s:.4f}")

print(f"\nBaseline (base MPNet): Spearman {baseline['spearman']:.4f} | MAE {baseline['mae']:.4f}")
sp_m, _ = agg('platt', 'spearman')
print(f"Fine-tuning delta (Platt): +{sp_m - baseline['spearman']:.4f} Spearman")

# Production seed: median Platt MAE (robust representative run, not the luckiest one)
maes = [r['platt']['mae'] for r in seed_runs]
prod = seed_runs[int(np.argsort(maes)[len(maes)//2])]
print(f"\nPRODUCTION SEED: {prod['seed']} (median Platt MAE {prod['platt']['mae']:.4f})")
print(f"  -> ships from {prod['model_path']}")


## 5. Platt vs isotonic — settle it with a bootstrap

The v2 reruns showed the two calibrators swapping places (~0.002 MAE apart). Instead of
declaring a winner from one draw of 106 test pairs, resample those pairs 2,000 times and
look at the distribution of the paired difference `MAE(Platt) − MAE(isotonic)`.

If the 95% CI straddles zero, they are statistically indistinguishable — and Platt ships
on the robustness argument (2 parameters can't overfit a 106-pair calibration split),
not on a coin-flip metric win.


In [ ]:
# ============================================================
# CELL 8: Bootstrap comparison (production seed)
# ============================================================
t_raw = np.asarray(prod['_test_raw'])
true_arr = np.asarray(test_true)
pred_platt = np.asarray(prod['_platt_obj'](t_raw.tolist()))
pred_iso   = np.asarray(prod['_iso_obj'].predict(t_raw.tolist()))

rng = np.random.default_rng(CONFIG["split_seed"])
n = len(true_arr)
diffs = []
for _ in range(CONFIG["bootstrap_resamples"]):
    idx = rng.integers(0, n, n)
    mae_p = np.mean(np.abs(true_arr[idx] - pred_platt[idx]))
    mae_i = np.mean(np.abs(true_arr[idx] - pred_iso[idx]))
    diffs.append(mae_p - mae_i)
diffs = np.asarray(diffs)

lo, hi = np.percentile(diffs, [2.5, 97.5])
point = float(np.mean(np.abs(true_arr - pred_platt)) - np.mean(np.abs(true_arr - pred_iso)))
print(f"MAE(Platt) - MAE(isotonic), point estimate: {point:+.4f}")
print(f"95% bootstrap CI: [{lo:+.4f}, {hi:+.4f}]  ({CONFIG['bootstrap_resamples']} resamples)")
print(f"P(Platt better): {float((diffs < 0).mean()):.2f}")

if lo <= 0 <= hi:
    print("\nVERDICT: CI straddles zero -> statistically TIED.")
    print("Platt ships on robustness (2 params can't overfit 106 calibration pairs).")
else:
    winner = 'Platt' if hi < 0 else 'Isotonic'
    print(f"\nVERDICT: {winner} is genuinely better on this test set.")

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].hist(diffs, bins=50, color='#534AB7', edgecolor='white')
axes[0].axvline(0, color='black', lw=1)
axes[0].axvline(lo, color='#A32D2D', ls='--', lw=1); axes[0].axvline(hi, color='#A32D2D', ls='--', lw=1)
axes[0].set_title('Bootstrap: MAE(Platt) - MAE(iso)', fontweight='bold')
axes[0].set_xlabel('difference (negative = Platt better)')

x = np.arange(len(seed_runs)); w = 0.35
axes[1].bar(x-w/2, [r['platt']['mae'] for r in seed_runs], w, label='Platt', color='#185FA5', edgecolor='white')
axes[1].bar(x+w/2, [r['isotonic']['mae'] for r in seed_runs], w, label='Isotonic', color='#0F6E56', edgecolor='white')
axes[1].set_xticks(x); axes[1].set_xticklabels([f"seed {r['seed']}" for r in seed_runs])
axes[1].set_ylabel('MAE (final test)'); axes[1].set_title('Per-seed calibrator MAE', fontweight='bold')
axes[1].legend()

axes[2].scatter(true_arr, pred_platt, alpha=0.55, s=28, c='#185FA5', edgecolors='white', linewidth=0.4)
axes[2].plot([0,1],[0,1],'k--',alpha=0.35)
axes[2].set_xlabel('True score'); axes[2].set_ylabel('Predicted (Platt)')
axes[2].set_title(f"Production seed {prod['seed']} — MAE {prod['platt']['mae']:.3f}", fontweight='bold')
axes[2].set_xlim(0,1); axes[2].set_ylim(0,1)

plt.tight_layout()
plt.savefig('production_v3_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: production_v3_analysis.png")


## 6. Recruiter metric — precision@1 on unseen postings

Spearman measures global ranking across all pairs, but a recruiter asks a narrower
question: *for THIS posting, did the system put the right resume on top?*

The external set has ~4 candidate resumes per JD. For every unseen JD, rank its
candidates by model score and check whether the top pick matches the ground-truth best.
Ranking is calibration-invariant (monotone maps preserve order), so raw cosine scores on
the full 212 external pairs are leakage-free — no calibration data is being reused.


In [ ]:
# ============================================================
# CELL 9: Precision@1 per unseen JD (production seed)
# ============================================================
prod_model = SentenceTransformer(prod['model_path'])
ext_full = ext_full.reset_index(drop=True)
ext_full['pred_raw'] = raw_scores(prod_model, ext_full)

hits, groups, margins = 0, 0, []
for jd_text, grp in ext_full.groupby('jd'):
    if len(grp) < 2 or grp['score'].nunique() < 2:
        continue  # can't rank a single candidate or all-equal truths
    groups += 1
    top_pred_idx = grp['pred_raw'].idxmax()
    best_true = grp['score'].max()
    if grp.loc[top_pred_idx, 'score'] >= best_true - 1e-9:
        hits += 1
    margins.append(float(best_true - grp.loc[top_pred_idx, 'score']))

p_at_1 = hits / groups
print(f"Unseen JDs with rankable candidate sets: {groups}")
print(f"Precision@1 (top-ranked resume is the true best): {p_at_1:.1%}")
print(f"Mean shortfall when wrong: {np.mean([m for m in margins if m > 0]) if any(m > 0 for m in margins) else 0:.3f} score points")

ranking_result = {"groups": int(groups), "precision_at_1": round(float(p_at_1), 4)}


## 7. Error analysis


In [ ]:
# ============================================================
# CELL 10: Error analysis by match type and industry (production seed, Platt)
# ============================================================
error_by_type(ext_test, pred_platt.tolist(), f"Production (seed {prod['seed']}, Platt) — final test")

a = ext_test.copy(); a['pred'] = pred_platt; a['err'] = abs(a['score'].astype(float) - a['pred'])
print("\n\nINDUSTRY ANALYSIS")
for ind in sorted(a['industry'].unique()):
    s = a[a['industry']==ind]; mae = s['err'].mean()
    icon = "OK " if mae < 0.15 else "WARN" if mae < 0.25 else "BAD "
    print(f"  [{icon}] {ind:<45}: MAE={mae:.4f} (n={len(s)})")

print("\n\nTOP 5 WORST ERRORS")
for _, row in a.nlargest(5, 'err').iterrows():
    print(f"  true {row['score']:.3f} -> pred {row['pred']:.3f} | {row['match_type']} | {row['industry']}")


## 8. Export everything

Saves the results JSON (this is the file to send back for re-analysis, along with the
executed notebook), both calibrators, and downloads them to your machine.


In [ ]:
# ============================================================
# CELL 11: Save results + calibrators, download
# ============================================================
results = {
    "config": {k: v for k, v in CONFIG.items()},
    "baseline_base_mpnet": baseline,
    "per_seed": [{k: v for k, v in r.items() if not k.startswith('_')} for r in seed_runs],
    "aggregate": rows_agg,
    "production_seed": prod['seed'],
    "production": {"raw": prod['raw'], "platt": prod['platt'], "isotonic": prod['isotonic']},
    "bootstrap": {
        "mae_diff_point": round(point, 4),
        "ci95": [round(float(lo), 4), round(float(hi), 4)],
        "p_platt_better": round(float((diffs < 0).mean()), 3),
        "verdict": "tied" if lo <= 0 <= hi else ("platt" if hi < 0 else "isotonic"),
    },
    "ranking": ranking_result,
}
with open('production_v3_results.json', 'w') as f:
    json.dump(results, f, indent=2)

with open('platt_calibrator.pkl', 'wb') as f:
    pickle.dump(prod['_platt_obj'], f)
with open('isotonic_calibrator.pkl', 'wb') as f:
    pickle.dump(prod['_iso_obj'], f)

print(json.dumps(results, indent=2))
files.download('production_v3_results.json')
files.download('platt_calibrator.pkl')
files.download('isotonic_calibrator.pkl')


## 9. Publish to HuggingFace Hub

This is what makes the live Streamlit demo serve the real model. Get a **write** token
from https://huggingface.co/settings/tokens. The model card is generated from this run's
actual numbers.


In [ ]:
# ============================================================
# CELL 12: Push model + calibrators + model card to HF Hub
# ============================================================
from huggingface_hub import login, HfApi
login()  # paste your WRITE token

REPO = CONFIG["hf_repo"]
prod_model.push_to_hub(REPO, exist_ok=True)

card = f"""---
license: apache-2.0
base_model: sentence-transformers/all-mpnet-base-v2
pipeline_tag: sentence-similarity
tags: [sentence-transformers, resume-matching, fine-tuned]
---

# ResumeAI — Resume / Job-Description Matcher (MPNet, fine-tuned)

Fine-tuned `all-mpnet-base-v2` scoring resume <-> job-description fit, trained with a
combined CoSENT + CosineSimilarity objective on 815 curated pairs from 305 real job
postings, with 3x data augmentation. Scores are calibrated with Platt scaling
(`platt_calibrator.pkl` in this repo) fitted on an external calibration split.

## Evaluation (106 fully held-out pairs from unseen postings)

| Metric | Value |
|---|---|
| Spearman (Platt, production seed {prod['seed']}) | {prod['platt']['spearman']:.4f} |
| MAE (Platt) | {prod['platt']['mae']:.4f} |
| Spearman across {len(CONFIG['seeds'])} seeds | {rows_agg['platt']['spearman_mean']:.4f} +/- {rows_agg['platt']['spearman_std']:.4f} |
| MAE across seeds | {rows_agg['platt']['mae_mean']:.4f} +/- {rows_agg['platt']['mae_std']:.4f} |
| Base-model baseline (Spearman) | {baseline['spearman']:.4f} |
| Precision@1 over {ranking_result['groups']} unseen postings | {ranking_result['precision_at_1']:.1%} |

## Usage

```python
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("{REPO}")
emb = model.encode([resume_text, jd_text])
raw = cosine_similarity([emb[0]], [emb[1]])[0][0]
# apply platt_calibrator.pkl for a calibrated 0-1 score
```

Research repo: https://github.com/dlepighe1/Resume-jd-matcher
"""

api = HfApi()
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=REPO)
api.upload_file(path_or_fileobj="platt_calibrator.pkl", path_in_repo="platt_calibrator.pkl", repo_id=REPO)
api.upload_file(path_or_fileobj="isotonic_calibrator.pkl", path_in_repo="isotonic_calibrator.pkl", repo_id=REPO)
api.upload_file(path_or_fileobj="production_v3_results.json", path_in_repo="production_v3_results.json", repo_id=REPO)

print(f"\nDone: https://huggingface.co/{REPO}")
print("The Streamlit app will now load this model automatically (MODEL_ID matches).")


## What to send back for re-analysis

1. **This notebook, executed** (`File → Download → .ipynb`)
2. **`production_v3_results.json`** (downloaded by Cell 11)

The repo's `Results/` and README get updated from those two files — including the
mean ± std headline, the bootstrap verdict, and precision@1.
